In [36]:
import pandas as pd

def top_entailment_per_target(df, entail = "ENTAILMENT"):
    """
    For each target, select the row with the highest 'entailment' score.
    
    Parameters:
        df (pd.DataFrame): DataFrame with columns including 'target' and 'entailment'.
        
    Returns:
        pd.DataFrame: DataFrame with one row per target, having the highest entailment score.
    """
    # For each target, find index of row with max entailment
    idx = df.groupby('target')[entail].idxmax(axis=0)
    return df.loc[idx].reset_index(drop=True)

# Example usage
# data = {
#     'textid': ['greeting', 'question', 'greeting', 'question','greeting', 'question'],
#     'target': [0, 0, 1, 1,2,2],
#     'model': ['huggingface/distilbert-base-uncased-finetuned-mnli']*6,
#     'tokenizer': ['huggingface/distilbert-base-uncased-finetuned-mnli']*6,
#     'predicted': ['entailment', 'entailment', 'contradiction', 'contradiction', 'contradiction', 'contradiction'],
#     'prob': [0.8553178310394287, 0.8553178310394287, 0.3848722577095032, 0.3848722577095032, 0.3848722577095032, 0.3848722577095032],
#     'entailment': [0.8553178310394287, 0.8553178310394287, 0.31023797392845154, 0.31023797392845154, 0.31023797392845154, 0.31023797392845154],
#     'neutral': [0.13772304356098175, 0.13772304356098175, 0.3048897087574005, 0.3048897087574005, 0.3048897087574005, 0.3048897087574005],
#     'contradiction': [0.006959038320928812, 0.006959038320928812, 0.3848722577095032, 0.3848722577095032, 0.3848722577095032, 0.3848722577095032]
# }

data = pd.read_csv("predictions/Copy_Backup/Shared_BaseTest_predictions.tsv",sep="\t")

# df = pd.DataFrame(data)
top_df = top_entailment_per_target(data)
display(top_df)



/var/folders/f8/58q303js66gc16m838wxn1fw0000gn/T/ipykernel_72788/3909059465.py:14: FutureWarning: The 'axis' keyword in SeriesGroupBy.idxmax is deprecated and will be removed in a future version. Call without passing 'axis' instead.
  idx = df.groupby('target')[entail].idxmax(axis=0)


,textid,target,model,tokenizer,predicted,prob,CONTRADICTION,NEUTRAL,ENTAILMENT
0,1,0,microsoft/deberta-base-mnli,microsoft/deberta-base-mnli,NEUTRAL,0.498680,0.427861,0.498680,0.073459
1,1,1,microsoft/deberta-base-mnli,microsoft/deberta-base-mnli,CONTRADICTION,0.884839,0.884839,0.052157,0.063004
2,3,2,microsoft/deberta-base-mnli,microsoft/deberta-base-mnli,NEUTRAL,0.832043,0.145391,0.832043,0.022566
3,1,3,microsoft/deberta-base-mnli,microsoft/deberta-base-mnli,NEUTRAL,0.916654,0.056780,0.916654,0.026565
4,2,4,microsoft/deberta-base-mnli,microsoft/deberta-base-mnli,NEUTRAL,0.828863,0.126318,0.828863,0.044819
...,...,...,...,...,...,...,...,...,...
7595,1,7595,microsoft/deberta-base-mnli,microsoft/deberta-base-mnli,CONTRADICTION,0.823755,0.823755,0.137614,0.038632
7596,2,7596,microsoft/deberta-base-mnli,microsoft/deberta-base-mnli,NEUTRAL,0.582558,0.402302,0.582558,0.015140
7597,2,7597,microsoft/deberta-base-mnli,microsoft/deberta-base-mnli,NEUTRAL,0.914520,0.019696,0.914520,0.065784
7598,3,7598,microsoft/deberta-base-mnli,microsoft/deberta-base-mnli,NEUTRAL,0.680466,0.164015,0.680466,0.155519


,textid,target,model,tokenizer,predicted,prob,CONTRADICTION,NEUTRAL,ENTAILMENT
0,0,0,microsoft/deberta-base-mnli,microsoft/deberta-base-mnli,CONTRADICTION,0.948134,0.948134,0.045721,0.006145
1,1,0,microsoft/deberta-base-mnli,microsoft/deberta-base-mnli,NEUTRAL,0.498680,0.427861,0.498680,0.073459
2,2,0,microsoft/deberta-base-mnli,microsoft/deberta-base-mnli,CONTRADICTION,0.695733,0.695733,0.270892,0.033375
3,3,0,microsoft/deberta-base-mnli,microsoft/deberta-base-mnli,NEUTRAL,0.717915,0.258338,0.717915,0.023747
4,0,1,microsoft/deberta-base-mnli,microsoft/deberta-base-mnli,CONTRADICTION,0.974294,0.974294,0.024323,0.001383
...,...,...,...,...,...,...,...,...,...
30395,3,7598,microsoft/deberta-base-mnli,microsoft/deberta-base-mnli,NEUTRAL,0.680466,0.164015,0.680466,0.155519
30396,0,7599,microsoft/deberta-base-mnli,microsoft/deberta-base-mnli,CONTRADICTION,0.717300,0.717300,0.268941,0.013759
30397,1,7599,microsoft/deberta-base-mnli,microsoft/deberta-base-mnli,CONTRADICTION,0.872717,0.872717,0.123776,0.003506
30398,2,7599,microsoft/deberta-base-mnli,microsoft/deberta-base-mnli,NEUTRAL,0.533551,0.453124,0.533551,0.013326


In [41]:
import pandas as pd
import numpy as np

def get_positive_example(df, percent=0.1, score_cols=['entailment', 'neutral', 'contradiction']):
    """
    From the top entailment per target, select the top X% most confident examples 
    based on the delta between the top and second-highest MNLI scores.
    
    Parameters:
        df (pd.DataFrame): DataFrame with MNLI score columns.
        percent (float): Fraction of top examples to return (0 < percent <= 1)
        score_cols (list): Names of MNLI score columns to consider
        
    Returns:
        pd.DataFrame: DataFrame with top X% most confident examples.
    """
    # First, select top entailment per target
    top_df = top_entailment_per_target(df)
    
    # Extract MNLI score values
    scores = top_df[score_cols]
    print(  scores)
    #Maximum values per row:
    largeset_score = np.max(scores,axis = 1)
    # make it so that 
    # all value before -2 are less thanit and all value after it are greater, so we specify that there is only index -1 greater than it
    second_largest = pd.Series([np.partition(scores.iloc[i], -2)[-2] for i in range(len(scores))])
    print("largest\n")
    print(largeset_score)
    print("second largest\n")
    print( second_largest )
    
    delta =  largeset_score-second_largest 
    print("delta\n")
    print( delta )
    
    # this gets us the value on the top
    
    top_df["delta"]= delta
    
    # the smaller index the greater
    df_sorted_delta = top_df.sort_values(by='delta', ascending=False)
    
    top_percent_df = df_sorted_delta[:int(len(df_sorted_delta)*percent)]
    
    top_percent_df["target"] =  len(df_sorted_delta)*percent * [score_cols[0]]

    return top_percent_df
    

    


# Only MNLI scores matter
mnli_labels = ['ENTAILMENT', 'NEUTRAL', 'CONTRADICTION']

# mnli_labels =['entailment', 'neutral', 'contradiction']



# Get top 10% most confident rows

print("positive\n")
display(get_positive_example(data, percent=1, score_cols=mnli_labels))

positive

      ENTAILMENT   NEUTRAL  CONTRADICTION
0       0.073459  0.498680       0.427861
1       0.063004  0.052157       0.884839
2       0.022566  0.832043       0.145391
3       0.026565  0.916654       0.056780
4       0.044819  0.828863       0.126318
...          ...       ...            ...
7595    0.038632  0.137614       0.823755
7596    0.015140  0.582558       0.402302
7597    0.065784  0.914520       0.019696
7598    0.155519  0.680466       0.164015
7599    0.013759  0.268941       0.717300

[7600 rows x 3 columns]
largest

0       0.498680
1       0.884839
2       0.832043
3       0.916654
4       0.828863
          ...   
7595    0.823755
7596    0.582558
7597    0.914520
7598    0.680466
7599    0.717300
Length: 7600, dtype: float64
second largest

0       0.427861
1       0.063004
2       0.145391
3       0.056780
4       0.126318
          ...   
7595    0.137614
7596    0.402302
7597    0.065784
7598    0.164015
7599    0.268941
Length: 7600, dtype: float64
delt

/var/folders/f8/58q303js66gc16m838wxn1fw0000gn/T/ipykernel_72788/3909059465.py:14: FutureWarning: The 'axis' keyword in SeriesGroupBy.idxmax is deprecated and will be removed in a future version. Call without passing 'axis' instead.
  idx = df.groupby('target')[entail].idxmax(axis=0)


,textid,target,model,tokenizer,predicted,prob,CONTRADICTION,NEUTRAL,ENTAILMENT,delta
2935,3,ENTAILMENT,microsoft/deberta-base-mnli,microsoft/deberta-base-mnli,NEUTRAL,0.997718,0.001808,0.997718,0.000474,0.995911
4984,0,ENTAILMENT,microsoft/deberta-base-mnli,microsoft/deberta-base-mnli,NEUTRAL,0.997498,0.002074,0.997498,0.000428,0.995424
7262,3,ENTAILMENT,microsoft/deberta-base-mnli,microsoft/deberta-base-mnli,CONTRADICTION,0.995330,0.995330,0.002889,0.001781,0.992442
1125,3,ENTAILMENT,microsoft/deberta-base-mnli,microsoft/deberta-base-mnli,NEUTRAL,0.995808,0.003583,0.995808,0.000609,0.992224
1098,1,ENTAILMENT,microsoft/deberta-base-mnli,microsoft/deberta-base-mnli,NEUTRAL,0.995013,0.003538,0.995013,0.001449,0.991475
...,...,...,...,...,...,...,...,...,...,...
6290,2,ENTAILMENT,microsoft/deberta-base-mnli,microsoft/deberta-base-mnli,NEUTRAL,0.408915,0.408404,0.408915,0.182681,0.000511
3169,2,ENTAILMENT,microsoft/deberta-base-mnli,microsoft/deberta-base-mnli,NEUTRAL,0.475624,0.475119,0.475624,0.049257,0.000505
1199,3,ENTAILMENT,microsoft/deberta-base-mnli,microsoft/deberta-base-mnli,CONTRADICTION,0.463286,0.463286,0.462831,0.073883,0.000455
7559,1,ENTAILMENT,microsoft/deberta-base-mnli,microsoft/deberta-base-mnli,ENTAILMENT,0.489015,0.022311,0.488675,0.489015,0.000340


In [ ]:
# this is just redefining the same function without the print so it is less annoying 

def get_positive_example(df, percent=0.1, score_cols=['ENTAILMENT', 'NEUTRAL', 'CONTRADICTION']):
    """
    From the top entailment per target, select the top X% most confident examples base on delta
    based on the delta between the top and second-highest MNLI scores.
    
    Parameters:
        df (pd.DataFrame): DataFrame with MNLI score columns.
        percent (float): Fraction of top examples to return (0 < percent <= 1)
        score_cols (list): Names of MNLI score columns to consider
        
    Returns:
        pd.DataFrame: DataFrame with top X% most confident examples.
    """
    # First, select top entailment per target
    top_df = top_entailment_per_target(df)
    
    # Extract MNLI score values
    scores = top_df[score_cols]
    #Maximum values per row:
    largeset_score = np.max(scores,axis = 1)
    # make it so that 
    # all value before -2 are less thanit and all value after it are greater, so we specify that there is only index -1 greater than it
    second_largest = pd.Series([np.partition(scores.iloc[i], -2)[-2] for i in range(len(scores))])
  
    
    delta =  largeset_score-second_largest 
  
    
    # this gets us the value on the top
    
    top_df["delta"]= delta
    
    
    
    
    # the smaller index the greater
    df_sorted_delta = top_df.sort_values(by='delta', ascending=False)
    
    top_percent_df = df_sorted_delta[:int(len(df_sorted_delta)*percent)]

    return top_percent_df
    

In [160]:
    
import random

def get_negative_random(data,percent=0.1,contra = "CONTRADICTION"):
    """
    For each entailment pair, generate a negative example by replacing the class
    in the hypothesis with a random different class, and assign the contradict label.
    
    Parameters:
    
       Positve df
        
    Returns:
       Full finetuning dataset
    """

    pos = get_positive_example(data,percent=percent).dropna().reset_index(drop=True)
    sample_space = set(pos["textid"].unique())
    for idx in range(len(pos)):
        pos.loc[idx, "target"] = contra 
        event1 =  {pos.loc[idx, "textid"]}
        pos.loc[idx, "textid"] = int(random.sample(list(sample_space - event1), 1)[0])
        mapping_mask =  {0:"World" ,1:"Sports",2:"Business",3:"Sci/Tech"}
        label_type = mapping_mask[int(pos.loc[idx, "textid"])]
        pos.loc[idx, "pair"] = f"This example is {label_type}"
    
      

    return pos
        

        # # pos["predicted"] = pd.sample([])
        # mapping_mask =  {0:"World" ,1:"Sports",2:"Business",3:"Sci/Tech"}
        # text_classfication_true["predicted"] =  Max_entailment["textid"].map(mapping_mask)

    # pd.concat([])
    
positive = get_positive_example(data, percent=1, score_cols=mnli_labels)

get_negative_random(positive,percent=1)
    


/var/folders/f8/58q303js66gc16m838wxn1fw0000gn/T/ipykernel_72788/3909059465.py:14: FutureWarning: The 'axis' keyword in SeriesGroupBy.idxmax is deprecated and will be removed in a future version. Call without passing 'axis' instead.
  idx = df.groupby('target')[entail].idxmax(axis=0)
/var/folders/f8/58q303js66gc16m838wxn1fw0000gn/T/ipykernel_72788/3909059465.py:14: FutureWarning: The 'axis' keyword in SeriesGroupBy.idxmax is deprecated and will be removed in a future version. Call without passing 'axis' instead.
  idx = df.groupby('target')[entail].idxmax(axis=0)
/var/folders/f8/58q303js66gc16m838wxn1fw0000gn/T/ipykernel_72788/3171698347.py:19: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'CONTRADICTION' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  pos.loc[idx, "target"] = contra


,textid,target,model,tokenizer,predicted,prob,CONTRADICTION,NEUTRAL,ENTAILMENT,delta,pair
0,0,CONTRADICTION,microsoft/deberta-base-mnli,microsoft/deberta-base-mnli,NEUTRAL,0.997718,0.001808,0.997718,0.000474,0.995911,This example is World
1,3,CONTRADICTION,microsoft/deberta-base-mnli,microsoft/deberta-base-mnli,NEUTRAL,0.997498,0.002074,0.997498,0.000428,0.995424,This example is Sci/Tech
2,2,CONTRADICTION,microsoft/deberta-base-mnli,microsoft/deberta-base-mnli,CONTRADICTION,0.995330,0.995330,0.002889,0.001781,0.992442,This example is Business
3,0,CONTRADICTION,microsoft/deberta-base-mnli,microsoft/deberta-base-mnli,NEUTRAL,0.995808,0.003583,0.995808,0.000609,0.992224,This example is World
4,3,CONTRADICTION,microsoft/deberta-base-mnli,microsoft/deberta-base-mnli,NEUTRAL,0.995013,0.003538,0.995013,0.001449,0.991475,This example is Sci/Tech
...,...,...,...,...,...,...,...,...,...,...,...
7595,0,CONTRADICTION,microsoft/deberta-base-mnli,microsoft/deberta-base-mnli,NEUTRAL,0.408915,0.408404,0.408915,0.182681,0.000511,This example is World
7596,3,CONTRADICTION,microsoft/deberta-base-mnli,microsoft/deberta-base-mnli,NEUTRAL,0.475624,0.475119,0.475624,0.049257,0.000505,This example is Sci/Tech
7597,0,CONTRADICTION,microsoft/deberta-base-mnli,microsoft/deberta-base-mnli,CONTRADICTION,0.463286,0.463286,0.462831,0.073883,0.000455,This example is World
7598,2,CONTRADICTION,microsoft/deberta-base-mnli,microsoft/deberta-base-mnli,ENTAILMENT,0.489015,0.022311,0.488675,0.489015,0.000340,This example is Business


In [145]:
originl_train_data = pd.read_csv("NLP_dataset/MNLI_formatted/Shared_Test_MNLI.tsv",sep="\t")
# display(originl_train_data)

originl_train_data["pair"]

0            This example is World
1           This example is Sports
2        This example is Bussiness
3         This example is Sci/Tech
4            This example is World
                   ...            
30395     This example is Sci/Tech
30396        This example is World
30397       This example is Sports
30398    This example is Bussiness
30399     This example is Sci/Tech
Name: pair, Length: 30400, dtype: object

In [162]:

def create_fintune_data(data,originl_train_data,percent,num_label = 4):
    
    pos = get_positive_example(data,percent)
    neg = get_negative_random(data,percent)
    
    org_pos = pos.index 
    mnli_index_pos = org_pos * num_label
    mapping_mask =  {0:"World" ,1:"Sports",2:"Business",3:"Sci/Tech"}
    # Map textid -> label and format each as a string
    
    # apply a mask and create template
    pos_pairs = pos["textid"].map(mapping_mask).apply(lambda x: f"This example is {x}")

    # Combine with neg pairs
    # 
    pair_series = pd.concat([pos_pairs, neg["pair"]], ignore_index=True)


    template = pos["textid"].map(mapping_mask),neg["pair"]
    output = pd.DataFrame({
        
        "textid": range(2 * len(pos)),
        "pair": pair_series,
        "text":  pd.concat([originl_train_data.loc[mnli_index_pos,"text"], originl_train_data.loc[mnli_index_pos,"text"],],ignore_index=True),
        
        "label": pd.concat([pd.Series(len(pos)*["ENTAILMENT"]), neg["target"]],ignore_index=True)
        
        
    })
    
    return output
    
output = create_fintune_data(data,originl_train_data,1)
display(output)

output.to_csv("test.tsv",sep="\t")



/var/folders/f8/58q303js66gc16m838wxn1fw0000gn/T/ipykernel_72788/3909059465.py:14: FutureWarning: The 'axis' keyword in SeriesGroupBy.idxmax is deprecated and will be removed in a future version. Call without passing 'axis' instead.
  idx = df.groupby('target')[entail].idxmax(axis=0)
/var/folders/f8/58q303js66gc16m838wxn1fw0000gn/T/ipykernel_72788/3909059465.py:14: FutureWarning: The 'axis' keyword in SeriesGroupBy.idxmax is deprecated and will be removed in a future version. Call without passing 'axis' instead.
  idx = df.groupby('target')[entail].idxmax(axis=0)
/var/folders/f8/58q303js66gc16m838wxn1fw0000gn/T/ipykernel_72788/3171698347.py:19: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'CONTRADICTION' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  pos.loc[idx, "target"] = contra


,textid,pair,text,label
0,0,This example is Sci/Tech,Palo Alto-based Hewlett-Packard Co. has bought...,ENTAILMENT
1,1,This example is World,Some of the nation #39;s largest daily newspap...,ENTAILMENT
2,2,This example is Sci/Tech,This week's TravelWatch column profiles Anangu...,ENTAILMENT
3,3,This example is Sci/Tech,Update: A partnership may be crucial for long-...,ENTAILMENT
4,4,This example is Sports,A different way of calculating the medal stand...,ENTAILMENT
...,...,...,...,...
15195,15195,This example is World,Donald Trump #39;s flagship casino business ye...,CONTRADICTION
15196,15196,This example is Sports,Insurgents exploded two car bombs at the gates...,CONTRADICTION
15197,15197,This example is World,South Africa has canceled a meeting with prose...,CONTRADICTION
15198,15198,This example is Sci/Tech,SPACE.com - Although winter officially begins ...,CONTRADICTION


In [93]:
print(output["pair"].unique())

['This example is World']


In [94]:
def create_fintune_data(data, originl_train_data, percent, num_label=4):
    pos = get_positive_example(data, percent)
    neg = get_negative_random(data)
    
    # Convert indices to list and multiply by num_label
    mnli_index_pos = (pos.index * num_label).tolist()
    mnli_index_neg = (neg.index * num_label).tolist()
    
    # Combine into a single list or Series
    textid = mnli_index_pos + mnli_index_neg
    
    # Build the DataFrame
    output = pd.DataFrame({
        "textid": textid,
        "text": pd.concat([
            originl_train_data.loc[mnli_index_pos, "text"],
            originl_train_data.loc[mnli_index_neg, "text"]
        ], ignore_index=True),
        "pair": pd.concat([
            originl_train_data.loc[mnli_index_pos, "pair"],
            originl_train_data.loc[mnli_index_neg, "pair"]
        ], ignore_index=True),
        "label": pd.concat([
            originl_train_data.loc[mnli_index_pos, "target"],
            originl_train_data.loc[mnli_index_neg, "target"]
        ], ignore_index=True)
    })
    
    display(output)
    return output

output = create_fintune_data(data,originl_train_data,1)


/var/folders/f8/58q303js66gc16m838wxn1fw0000gn/T/ipykernel_72788/3909059465.py:14: FutureWarning: The 'axis' keyword in SeriesGroupBy.idxmax is deprecated and will be removed in a future version. Call without passing 'axis' instead.
  idx = df.groupby('target')[entail].idxmax(axis=0)
/var/folders/f8/58q303js66gc16m838wxn1fw0000gn/T/ipykernel_72788/3909059465.py:14: FutureWarning: The 'axis' keyword in SeriesGroupBy.idxmax is deprecated and will be removed in a future version. Call without passing 'axis' instead.
  idx = df.groupby('target')[entail].idxmax(axis=0)
/var/folders/f8/58q303js66gc16m838wxn1fw0000gn/T/ipykernel_72788/1687901979.py:19: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'CONTRADICTION' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  pos.loc[idx, "target"] = contra


,textid,text,pair,label
0,11740,Palo Alto-based Hewlett-Packard Co. has bought...,This example is World,2935
1,19936,Some of the nation #39;s largest daily newspap...,This example is World,4984
2,29048,This week's TravelWatch column profiles Anangu...,This example is World,7262
3,4500,Update: A partnership may be crucial for long-...,This example is World,1125
4,4392,A different way of calculating the medal stand...,This example is World,1098
...,...,...,...,...
15195,30380,Ukrainian presidential candidate Viktor Yushch...,This example is World,7595
15196,30384,With the supply of attractive pitching options...,This example is World,7596
15197,30388,Like Roger Clemens did almost exactly eight ye...,This example is World,7597
15198,30392,SINGAPORE : Doctors in the United States have ...,This example is World,7598
